In [1]:
import pandas as pd

In [2]:
# 1. Currency & Symbols Removal (Cleaning)
def clean_currency(value):
    if isinstance(value, str):
        return value.replace('$', '').replace(',', '').strip()
    return value

def clean_percentage(value):
    if isinstance(value, str):
        return value.replace('%', '').strip()
    return value

In [3]:
# 2. Market Cap Conversion (Converting T, B, M to Numbers)
def convert_suffix_to_number(value):
    if isinstance(value, str):
        value = value.replace('$', '').strip()
        if 'T' in value:
            return float(value.replace('T', '')) * 1_000_000_000_000
        elif 'B' in value:
            return float(value.replace('B', '')) * 1_000_000_000
        elif 'M' in value:
            return float(value.replace('M', '')) * 1_000_000
    return value

In [4]:
# Data Loading
df = pd.read_csv('coinmarketcap_data_selenium_bs4.csv')

In [5]:
df

,Rank,Name,Price,1h%,24h%,7d%,Market Cap,Volume(24h)
0,1,Bitcoin,"$79,655.86",0.99%,1.24%,1.83%,$1.59T,"$45,094,955,140"
1,2,Ethereum,"$2,355.82",0.80%,1.35%,1.21%,$284.02B,"$22,402,210,819"
2,3,Tether,$0.9997,0.01%,0.01%,0.04%,$189.53B,"$138,793,308,342"
3,4,XRP,$1.40,0.37%,0.57%,0.47%,$86.63B,"$2,231,367,497"
4,5,BNB,$625.76,0.35%,0.99%,0.36%,$84.34B,"$2,107,281,764"
...,...,...,...,...,...,...,...,...
195,196,Safe,$0.1418,0.20%,2.35%,1.99%,$105.04M,"$3,376,243"
196,197,Gas,$1.61,0.16%,0.78%,3.02%,$104.97M,"$3,088,461"
197,198,Axelar,$0.08917,1.00%,21.69%,59.88%,$103.76M,"$78,482,269"
198,199,Irys,$0.04085,0.06%,5.58%,21.85%,$104.87M,"$8,167,782"


In [6]:
# Applying Transformations
df_cleaned = df.copy()

In [7]:
# Clean Price and Volume
df_cleaned['Price'] = df_cleaned['Price'].apply(clean_currency).astype(float)
df_cleaned['Volume(24h)'] = df_cleaned['Volume(24h)'].apply(clean_currency).astype(float)

In [8]:
# Clean Percentage Columns
percentage_cols = ['1h%', '24h%', '7d%']
for col in percentage_cols:
    df_cleaned[col] = df_cleaned[col].apply(clean_percentage).astype(float)

In [9]:
# Clean Market Cap
df_cleaned['Market Cap'] = df_cleaned['Market Cap'].apply(convert_suffix_to_number).astype(float)

In [10]:
# 3. Feature Engineering (Best Transformations)
# Calculation: Volume to Market Cap Ratio (Liquidity check)
df_cleaned['Vol_MC_Ratio'] = df_cleaned['Volume(24h)'] / df_cleaned['Market Cap']

In [11]:
# Categorization: Large Cap vs Small Cap
df_cleaned['Cap_Category'] = pd.cut(df_cleaned['Market Cap'], 
                                   bins=[0, 1e9, 10e9, float('inf')], 
                                   labels=['Small Cap', 'Mid Cap', 'Large Cap'])

In [14]:
df_cleaned.head()

,Rank,Name,Price,1h%,24h%,7d%,Market Cap,Volume(24h),Vol_MC_Ratio,Cap_Category
0,1,Bitcoin,79655.8600,0.99,1.24,1.83,1.590000e+12,4.509496e+10,0.028362,Large Cap
1,2,Ethereum,2355.8200,0.80,1.35,1.21,2.840200e+11,2.240221e+10,0.078875,Large Cap
2,3,Tether,0.9997,0.01,0.01,0.04,1.895300e+11,1.387933e+11,0.732303,Large Cap
3,4,XRP,1.4000,0.37,0.57,0.47,8.663000e+10,2.231367e+09,0.025757,Large Cap
4,5,BNB,625.7600,0.35,0.99,0.36,8.434000e+10,2.107282e+09,0.024986,Large Cap


In [13]:
df.head()

,Rank,Name,Price,1h%,24h%,7d%,Market Cap,Volume(24h)
0,1,Bitcoin,"$79,655.86",0.99%,1.24%,1.83%,$1.59T,"$45,094,955,140"
1,2,Ethereum,"$2,355.82",0.80%,1.35%,1.21%,$284.02B,"$22,402,210,819"
2,3,Tether,$0.9997,0.01%,0.01%,0.04%,$189.53B,"$138,793,308,342"
3,4,XRP,$1.40,0.37%,0.57%,0.47%,$86.63B,"$2,231,367,497"
4,5,BNB,$625.76,0.35%,0.99%,0.36%,$84.34B,"$2,107,281,764"


In [15]:
df_cleaned.to_csv('coinmarketcap_data_transformed.csv', index=False)